# Training — ML and ML-Lite surrogates

Depth-scaled U-Net surrogates for the nonlinear four-wave interaction `S_nl`, trained on deep-water WRT labels.

- **ML-Lite**: 1-level U-Net, `base_ch=16`
- **ML**: 2-level U-Net, `base_ch=32`

Pipeline: load data -> preprocess/normalize -> train.

## 1. Load data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = "data_train_deep/"  # folder with the training CSVs

base_path = DATA_DIR  # folder with efth_*.csv, snl_*.csv, depth_*.csv #kept 1 percent of depth>998
data_input = pd.concat([
    pd.read_csv(f'{base_path}efth_d20_998_999m.csv', header=None),
    
], ignore_index=True)
data_snl = pd.concat([
    pd.read_csv(f'{base_path}snl_d20_998_999m.csv', header=None),
], ignore_index=True)
data_depth = pd.concat([
    pd.read_csv(f'{base_path}depth_d20_998_999m.csv', header=None),
], ignore_index=True)

x_train_s = np.reshape(data_input, (data_input.shape[0], 24, 40, 1))
y_train_s = np.reshape(data_snl, (data_snl.shape[0], 24, 40, 1))
x_train_s=x_train_s[:, :, :40, :]
y_train_s=y_train_s[:, :, :40, :]
print(x_train_s.shape)
x = np.asarray(x_train_s)
y = np.asarray(y_train_s)
depth = np.asarray(data_depth).reshape(-1)

tol = 1e-18
min_tail_len = 5
min_zero_dirs = 24

spec = x[..., 0]
N, ndirs, nf = spec.shape

is_zero = np.abs(spec) <= tol
zero_dir_count = np.sum(is_zero, axis=1)
freq_is_zero = zero_dir_count >= min_zero_dirs

bad_mask = np.array([
    any(np.all(freq_is_zero[i, j:]) and np.any(~freq_is_zero[i, :j])
        for j in range(nf - min_tail_len + 1))
    for i in range(N)
])
drop_start = np.array([
    next((j for j in range(nf - min_tail_len + 1)
          if np.all(freq_is_zero[i, j:]) and np.any(~freq_is_zero[i, :j])), -1)
    for i in range(N)
])

print(f"Total samples      : {N}")
print(f"Removed samples    : {bad_mask.sum()}")
print(f"Remaining samples  : {(~bad_mask).sum()}")

if bad_mask.sum() > 0:
    bad_idx = np.where(bad_mask)[0]
    print("\nFirst 5 removed samples:")
    for k, i in enumerate(bad_idx[:5], 1):
        print(f"#{k:02d} idx={i}, depth={depth[i]:.3f} m, zero-tail starts at freq bin {drop_start[i]}")

x_train_s = x_train_s[~bad_mask]
y_train_s = y_train_s[~bad_mask]
data_depth = depth[~bad_mask]

print("\nClean shapes:")
print("x_train_s_clean:", x_train_s.shape)
print("y_train_s_clean:", y_train_s.shape)
print("data_depth_clean:", data_depth.shape)

## 2. Preprocess and normalize

Input: `Q = f^11 * F^3`, peak-normalized and direction-centered. Target: `S_nl` scaled by `F_n^3 * f_n^11`, then a global std-scaling on both.

In [ ]:
# CONFIG
# ============================================================
NDIR = 24
NFREQ = 40
G_CONST = 9.81
FREQ0 = 0.03453
FREQ_GROWTH = 1.1
THETA0 = 7.5
DTHETA_DEG = 15.0
EPS = 1e-16

# ============================================================
# DATA SPLIT
# ============================================================
x_train, x_vali, y_train, y_vali = train_test_split(
    x_train_s,
    y_train_s,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

x_train = x_train.reshape((-1, NDIR, NFREQ, 1))
x_vali  = x_vali.reshape((-1, NDIR, NFREQ, 1))
y_train = y_train.reshape((-1, NDIR, NFREQ, 1))
y_vali  = y_vali.reshape((-1, NDIR, NFREQ, 1))

x_train_ori = x_train.copy()
x_vali_ori  = x_vali.copy()
y_train_ori = y_train.copy()
y_vali_ori  = y_vali.copy()

depth_all = np.asarray(data_depth).reshape(-1)
depth_train, depth_vali = train_test_split(
    depth_all,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("x_train_ori.shape:", x_train_ori.shape)
print("y_train_ori.shape:", y_train_ori.shape)
print("depth_train.shape:", depth_train.shape)
print("depth_vali.shape :", depth_vali.shape)


# ============================================================
# GRID
# ============================================================
f = np.zeros(NFREQ, dtype=np.float64)
f[0] = FREQ0
for n in range(1, NFREQ):
    f[n] = f[n - 1] * FREQ_GROWTH

theta = np.zeros(NDIR, dtype=np.float64)
theta[0] = THETA0
for n in range(1, NDIR):
    theta[n] = THETA0 + DTHETA_DEG * n

dtheta = np.deg2rad(DTHETA_DEG)
df = np.gradient(f)
omega = 2.0 * np.pi * f

# ============================================================
# PHYSICS HELPERS
# ============================================================
def solve_k(omega_in, depth_in, g=9.81, niter=30):
    omega_in = np.asarray(omega_in, dtype=np.float64)[None, :]
    depth_in = np.asarray(depth_in, dtype=np.float64).reshape(-1, 1)

    k = (omega_in ** 2) / g
    k = np.where(k <= 0.0, 1e-12, k)

    for _ in range(niter):
        kd = k * depth_in
        tanh_kd = np.tanh(kd)
        fval = g * k * tanh_kd - omega_in ** 2
        dfval = g * (tanh_kd + kd * (1.0 - tanh_kd ** 2))
        k = k - fval / np.maximum(dfval, 1e-12)

    return k


def compute_kdp_from_spectrum(x_in, depth_in, f_in, dtheta_in, g=9.81):
    E = x_in[..., 0]
    depth_in = np.asarray(depth_in, dtype=np.float64).reshape(-1)

    E1 = np.sum(E, axis=1) * dtheta_in
    ip = np.argmax(E1, axis=1)
    fp = f_in[ip]

    omega_in = 2.0 * np.pi * f_in
    k = solve_k(omega_in, depth_in, g=g, niter=30)
    kd = k * depth_in[:, None]

    kp = k[np.arange(k.shape[0]), ip]
    kdp = kd[np.arange(kd.shape[0]), ip]

    return kp, kdp, fp, ip


def compute_R_with_limiter(kdp, eps=1e-12, kdp_min=0.5):
    Csh1 = 5.5
    Csh2 = 5.0 / 6.0
    Csh3 = -5.0 / 4.0

    kdp = np.asarray(kdp, dtype=np.float64).reshape(-1)
    kdp_eff = np.maximum(kdp, kdp_min)

    R = (
        1.0
        + (Csh1 / kdp_eff)
        * (1.0 - Csh2 * kdp_eff)
        * np.exp(Csh3 * kdp_eff)
    )
    R = np.maximum(R, eps)
    return R, kdp_eff


# ============================================================
# NORMALIZATION
# INPUT:
#   x_raw_norm = Q / max(Q)
#   x_final    = x_raw_norm / x_scale
#
# OUTPUT:
#   y_raw_norm = Y / (R * F_n^3 * f_n^11)
#   y_final    = y_raw_norm / y_scale
# ============================================================
def normalize_dataset(x_in, y_in, f_in, R_in, eps=1e-16, center_dir=True):
    n_samples = x_in.shape[0]
    n_dir = x_in.shape[1]
    dir_center = n_dir // 2   # target index for the peak direction (= 12 for 24 dirs)

    x_raw_norm = np.zeros_like(x_in, dtype=np.float32)
    y_raw_norm = np.zeros_like(y_in, dtype=np.float32)

    f_indices = np.zeros(n_samples, dtype=np.int32)
    f_values = np.zeros(n_samples, dtype=np.float64)
    F_values = np.zeros(n_samples, dtype=np.float64)
    k_indices = np.zeros(n_samples, dtype=np.int32)
    R_values = np.zeros(n_samples, dtype=np.float64)
    dir_shifts = np.zeros(n_samples, dtype=np.int32)

    f_mat = f_in[None, :]

    for i in range(n_samples):
        G = x_in[i, :, :, 0].astype(np.float64)
        Y = y_in[i, :, :, 0].astype(np.float64)

        Q = (f_mat ** 11) * (G ** 3)

        k_idx, f_idx = np.unravel_index(np.argmax(Q), Q.shape)
        f_n = float(f_in[f_idx])
        F_n = float(G[k_idx, f_idx])
        R_n = float(R_in[i])

        F_n_safe = max(F_n, eps)
        f_n_safe = max(f_n, eps)

        x_norm_i = Q / max(float(np.max(Q)), eps)
        y_norm_i = Y / ((F_n_safe ** 3) * (f_n_safe ** 11))

        # Center the direction axis so the input peak (k_idx) lands at dir_center.
        # Same shift applied to x and y. Undone in denorm_y_sample / denorm_y_batch.
        if center_dir:
            shift = int(dir_center - k_idx)
            x_norm_i = np.roll(x_norm_i, shift, axis=0)
            y_norm_i = np.roll(y_norm_i, shift, axis=0)
        else:
            shift = 0

        x_raw_norm[i, :, :, 0] = x_norm_i.astype(np.float32)
        y_raw_norm[i, :, :, 0] = y_norm_i.astype(np.float32)

        f_indices[i] = f_idx
        f_values[i] = f_n
        F_values[i] = F_n
        k_indices[i] = k_idx
        R_values[i] = R_n
        dir_shifts[i] = shift

    return (
        x_raw_norm,
        y_raw_norm,
        f_indices,
        f_values,
        F_values,
        k_indices,
        R_values,
        dir_shifts,
    )


def denorm_y_sample(y_norm_2d, idx, y_scale, R_used, F_vals, f_vals, dir_shifts=None, eps=1e-16):
    """
    Inverse of normalize_dataset for one sample.
    1) multiply by y_scale
    2) undo direction roll  (np.roll by -shift along direction axis)
    3) multiply by R * F^3 * f^11
    """
    y_rnorm = np.asarray(y_norm_2d, dtype=np.float64) * y_scale

    if dir_shifts is not None:
        shift = int(dir_shifts[idx])
        if shift != 0:
            y_rnorm = np.roll(y_rnorm, -shift, axis=0)

    R = max(float(R_used[idx]), eps)
    F = max(float(F_vals[idx]), eps)
    f0 = max(float(f_vals[idx]), eps)

    return y_rnorm * (R * (F ** 3) * (f0 ** 11))


def denorm_y_batch(y_norm, y_scale, R_used, F_vals, f_vals, dir_shifts=None, eps=1e-16):
    """
    Batched inverse of normalize_dataset.
    y_norm: (N, NDIR, NFREQ) or (N, NDIR, NFREQ, 1)
    returns: same shape as input, in physical Snl units.
    """
    arr = np.asarray(y_norm, dtype=np.float64)
    squeeze_last = (arr.ndim == 4 and arr.shape[-1] == 1)
    if squeeze_last:
        arr = arr[..., 0]

    out = arr * y_scale

    if dir_shifts is not None:
        ds = np.asarray(dir_shifts, dtype=np.int64)
        for i in range(out.shape[0]):
            s = int(ds[i])
            if s != 0:
                out[i] = np.roll(out[i], -s, axis=0)

    R = np.maximum(np.asarray(R_used, dtype=np.float64), eps)
    F = np.maximum(np.asarray(F_vals, dtype=np.float64), eps)
    f0 = np.maximum(np.asarray(f_vals, dtype=np.float64), eps)
    scale = R * (F ** 3) * (f0 ** 11)

    out = out * scale[:, None, None]

    if squeeze_last:
        out = out[..., None]
    return out


# ============================================================
# COMPUTE DEPTH-RELATED METADATA
# ============================================================
kp_train, kdp_train, fp_train_phys, ip_train_phys = compute_kdp_from_spectrum(
    x_train_ori, depth_train, f, dtheta, g=G_CONST
)
kp_vali, kdp_vali, fp_vali_phys, ip_vali_phys = compute_kdp_from_spectrum(
    x_vali_ori, depth_vali, f, dtheta, g=G_CONST
)

R_train, kdp_eff_train = compute_R_with_limiter(kdp_train)
R_vali, kdp_eff_vali = compute_R_with_limiter(kdp_vali)

print("R_train stats:", np.nanmin(R_train), np.nanmean(R_train), np.nanmax(R_train))
print("R_vali  stats:", np.nanmin(R_vali), np.nanmean(R_vali), np.nanmax(R_vali))

# ============================================================
# APPLY NORMALIZATION
# ============================================================
(
    x_train_raw_norm,
    y_train_raw_norm,
    f_idx_train,
    f_val_train,
    F_val_train,
    k_idx_train,
    R_used_train,
    dir_shift_train,
) = normalize_dataset(x_train_ori, y_train_ori, f, R_train, eps=EPS, center_dir=True)

(
    x_vali_raw_norm,
    y_vali_raw_norm,
    f_idx_vali,
    f_val_vali,
    F_val_vali,
    k_idx_vali,
    R_used_vali,
    dir_shift_vali,
) = normalize_dataset(x_vali_ori, y_vali_ori, f, R_vali, eps=EPS, center_dir=True)

# ============================================================
# FINAL GLOBAL SCALING
# ============================================================
x_scale = max(float(np.std(x_train_raw_norm[:, :, :, 0])), 1e-8)
y_scale = max(float(np.std(y_train_raw_norm[:, :, :, 0])), 1e-8)

x_train = np.empty_like(x_train_raw_norm, dtype=np.float32)
x_vali  = np.empty_like(x_vali_raw_norm, dtype=np.float32)
y_train = np.empty_like(y_train_raw_norm, dtype=np.float32)
y_vali  = np.empty_like(y_vali_raw_norm, dtype=np.float32)

x_train[:, :, :, 0] = x_train_raw_norm[:, :, :, 0] / x_scale
x_vali[:, :, :, 0]  = x_vali_raw_norm[:, :, :, 0] / x_scale

y_train[:, :, :, 0] = y_train_raw_norm[:, :, :, 0] / y_scale
y_vali[:, :, :, 0]  = y_vali_raw_norm[:, :, :, 0] / y_scale

print("x_scale:", x_scale)
print("y_scale:", y_scale)
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

## 3. Model definition (U-Net)

Circular padding along direction, replicate along frequency.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))
print('device:', device)

H_IN, W_IN = 24, 40

class CircularPad2D_Torch(nn.Module):
    """
    Circular pad along H (direction, periodic), replicate along W (frequency).
    """
    def __init__(self, pad_dir=1, pad_freq=1):
        super().__init__()
        self.pad_dir = int(pad_dir)
        self.pad_freq = int(pad_freq)

    def forward(self, x):
        pd, pf = self.pad_dir, self.pad_freq
        if pd > 0:
            x = F.pad(x, (0, 0, pd, pd), mode="circular")
        if pf > 0:
            x = F.pad(x, (pf, pf, 0, 0), mode="replicate")
        return x

# ============================================================
# BLOCKS
# ============================================================
class DoubleConv_OnePad(nn.Module):
    """
    One padding call for two 3x3 valid convolutions.
    """
    def __init__(self, in_ch, out_ch, pad_dir=1, pad_freq=1):
        super().__init__()
        self.pad = CircularPad2D_Torch(pad_dir=2 * pad_dir, pad_freq=2 * pad_freq)
        self.c1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=0, bias=True)
        self.c2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=0, bias=True)

    def forward(self, x):
        x = self.pad(x)
        x = self.c1(x)
        x = F.relu(x, inplace=True)
        x = self.c2(x)
        x = F.relu(x, inplace=True)
        return x


class DownFast(nn.Module):
    def __init__(self, in_ch, out_ch, pad_dir=1, pad_freq=1):
        super().__init__()
        self.conv = DoubleConv_OnePad(in_ch, out_ch, pad_dir, pad_freq)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        f = self.conv(x)
        p = self.pool(f)
        return f, p


class UpFast(nn.Module):
    """
    nearest upsample (scale=2) -> 1x1 projection -> concat -> doubleconv.
    """
    def __init__(self, in_ch, skip_ch, out_ch, pad_dir=1, pad_freq=1):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1, padding=0, bias=True)
        self.conv = DoubleConv_OnePad(out_ch + skip_ch, out_ch, pad_dir, pad_freq)

    def forward(self, x, skip):
        x = self.up(x)
        x = self.proj(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x


class UNet_Faster_24x40(nn.Module):
    """
    x: (N,1,24,50)
    y: (N,1,24,50)

    1 pool level: 24x50 -> 12x25.
    Do NOT extend to 2 levels: 25/2 would be non-integer.
    """
    def __init__(self, base_ch=16, pad_dir=1, pad_freq=1):
        super().__init__()

        c1 = base_ch
        c2 = base_ch * 2

        self.down1 = DownFast(1, c1, pad_dir, pad_freq)

        self.bottleneck = DoubleConv_OnePad(c1, c2, pad_dir, pad_freq)

        self.up1 = UpFast(c2, c1, c1, pad_dir, pad_freq)

        self.out = nn.Conv2d(c1, 1, kernel_size=1, padding=0, bias=True)

    def forward(self, x):
        f1, p1 = self.down1(x)

        b = self.bottleneck(p1)

        u1 = self.up1(b, f1)

        return self.out(u1)


class UNet_Faster_24x40_Deep(nn.Module):
    """
    2 pool levels: 24x40 -> 12x20 -> 6x10.
    """
    def __init__(self, base_ch=32, pad_dir=1, pad_freq=1):
        super().__init__()

        c1 = base_ch
        c2 = base_ch * 2
        c3 = base_ch * 4

        self.down1 = DownFast(1, c1, pad_dir, pad_freq)
        self.down2 = DownFast(c1, c2, pad_dir, pad_freq)

        self.bottleneck = DoubleConv_OnePad(c2, c3, pad_dir, pad_freq)

        self.up2 = UpFast(c3, c2, c2, pad_dir, pad_freq)
        self.up1 = UpFast(c2, c1, c1, pad_dir, pad_freq)

        self.out = nn.Conv2d(c1, 1, kernel_size=1, padding=0, bias=True)

    def forward(self, x):
        f1, p1 = self.down1(x)
        f2, p2 = self.down2(p1)

        b = self.bottleneck(p2)

        u2 = self.up2(b, f2)
        u1 = self.up1(u2, f1)

        return self.out(u1)

## 4. Prepare tensors

In [ ]:
def to_nchw(x):
    """
    (N,H,W,1) -> (N,1,H,W), or pass through if already (N,1,H,W).
    Returns torch.float32 tensor on CPU.
    """
    if isinstance(x, torch.Tensor):
        t = x.detach()
        if t.ndim == 4 and t.shape[-1] == 1:
            t = t.permute(0, 3, 1, 2).contiguous()
        return t.float().cpu()

    x = np.asarray(x)
    if x.ndim == 4 and x.shape[-1] == 1:
        return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous().float()
    if x.ndim == 4 and x.shape[1] == 1:
        return torch.from_numpy(x).contiguous().float()

    raise ValueError(f"Expected (N,H,W,1) or (N,1,H,W), got {x.shape}")

Xtr, Ytr = to_nchw(x_train).to(device), to_nchw(y_train).to(device)
Xva, Yva = to_nchw(x_vali).to(device),  to_nchw(y_vali).to(device)
print('Xtr', tuple(Xtr.shape), '| Xva', tuple(Xva.shape))

## 5. Training loop

In [ ]:
def train_model(model, epochs=100, batch_size=64, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    N, Nv = Xtr.shape[0], Xva.shape[0]
    best_val, best_state = float('inf'), None
    for epoch in range(1, epochs + 1):
        model.train(); perm = torch.randperm(N, device=device); tr = 0.0
        for b0 in range(0, N, batch_size):
            sel = perm[b0:b0 + batch_size]
            xb, yb = Xtr.index_select(0, sel), Ytr.index_select(0, sel)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
            tr += loss.item() * xb.size(0)
        model.eval(); va = 0.0
        with torch.inference_mode():
            for b0 in range(0, Nv, batch_size):
                xb, yb = Xva[b0:b0 + batch_size], Yva[b0:b0 + batch_size]
                va += loss_fn(model(xb), yb).item() * xb.size(0)
        tr, va = tr / N, va / Nv
        print(f'epoch {epoch:03d}/{epochs}  train {tr:.4e}  val {va:.4e}')
        if va < best_val:
            best_val = va
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.eval()


def export(model, tag):
    torch.save(model.state_dict(), f'{tag}_state_dict.pt')
    torch.jit.trace(model.cpu(), torch.randn(1, 1, H_IN, W_IN)).save(f'{tag}_torchscript.pt')
    print('saved', tag)

## 6. Train ML-Lite  (1-level, base_ch=16)

In [ ]:
ml_lite = train_model(UNet_Faster_24x40(base_ch=16), epochs=100, batch_size=64)
export(ml_lite, 'unet_faster_24x40_base16')

## 7. Train ML  (2-level deep, base_ch=32)

In [ ]:
ml = train_model(UNet_Faster_24x40_Deep(base_ch=32), epochs=100, batch_size=128)
export(ml, 'unet_faster_24x40_base32_deep')